# Silver Transforms — From Bronze to Analytically Usable

This notebook transforms the four bronze tables into six silver tables. Bronze landed raw data with explicit types — useful as a stable foundation, not directly queryable for the business question. Silver is where I make the modeling decisions that turn it into something analytically usable.

**Why I'm separating silver out at all:**

The MUP-PHY bronze table has one row per NPI × HCPCS × place-of-service. That means a single cardiologist who performed 30 different procedures has 30 rows. To answer "what did this provider cost Medicare?" I have to aggregate down to NPI. That aggregation is one of dozens of similar decisions silver has to make. Doing them once here means downstream gold tables and analyses don't each have to reinvent the same logic.

**What silver produces:**

| Silver table | Grain | Source |
|---|---|---|
| `silver.providers` | one row per NPI | aggregated from `bronze.medicare_physician_payments` |
| `silver.provider_services` | one row per NPI × HCPCS × place-of-service | cleaned `bronze.medicare_physician_payments` |
| `silver.hospitals` | one row per CCN | cleaned `bronze.hospital_info` |
| `silver.hospital_quality` | one row per CCN × measure | unified from the three quality bronze tables |
| `silver.npi_to_ccn_bridge` | one row per matched NPI ↔ CCN pair | derived via fuzzy matching between providers and hospitals |
| `silver.specialty_lookup` | one row per specialty (only if needed) | derived from MUP-PHY specialty values |

**How each section is structured:**

I'm writing a markdown cell before each silver table that lays out: what the table produces, what grain I'm picking, what alternatives I considered, why I chose the one I did, and what the chosen approach gives up. I think this matters because the "what" of silver code is rarely interesting on its own — the interesting part is why I chose this aggregation over that one, this match algorithm over the alternative. The markdown cells are where that thinking lives.

**Preview of the hardest decision:**

The most interesting modeling problem here is connecting physician data (keyed on NPI) to hospital data (keyed on CCN). The source data doesn't have a clean join between them. I'm using address-based fuzzy matching in `silver.npi_to_ccn_bridge` — I'll explain my reasoning, the alternatives I considered, and the match confidence approach when I get to that section.

In [0]:
# Setup: imports and constants

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType, DateType, TimestampType
)
from pyspark.sql.window import Window

# Catalog and schema constants
CATALOG = "medicare_provider_quality"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Set current catalog and the *silver* schema as default for writes
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SILVER_SCHEMA}")

print(f"Catalog:        {CATALOG}")
print(f"Default schema: {SILVER_SCHEMA}")
print(f"Bronze schema:  {BRONZE_SCHEMA}  (read-only source)")

Catalog:        medicare_provider_quality
Default schema: silver
Bronze schema:  bronze  (read-only source)


## 1. silver.providers — one row per NPI

This is the central table for the project's provider-level analysis. Every downstream question about "what did Medicare pay this doctor?" or "which specialties are most expensive?" reads from this table.

### What this produces

One row per NPI (National Provider Identifier), with rolled-up totals across every HCPCS (Healthcare Common Procedure Coding System) service that provider billed in 2023:

- Identifying info (name, credentials, individual vs organization flag)
- Practice location (chosen state, city, ZIP — see below)
- Specialty (chosen — see below)
- Spending aggregates (total Medicare payment, total allowed amount, total submitted charges)
- Service volume aggregates (total services, unique beneficiaries served, unique HCPCS codes billed)

### The modeling decisions

The bronze table has one row per NPI × HCPCS × place-of-service. To collapse to one row per NPI, I have to choose what to do when a provider's rows disagree with each other. The two real questions:

**Question 1: What state/city/ZIP do I assign to a provider who bills from multiple locations?**

About 8% of NPIs in MUP-PHY have services billed across more than one state. I considered three approaches:

- **Use the location with the most services billed.** Reflects where the provider actually works most. This is what I'm going with.
- **Use the location of the highest-paid service.** Skewed by outliers — one $20K procedure could flip an entire provider's "location" away from their actual home base.
- **Keep all locations as an array.** Cleaner from a data-integrity perspective, but every downstream query has to explode it back out, and 92% of providers only have one location anyway. Not worth the complexity.

**Question 2: What specialty do I assign to a provider who shows up under multiple specialty values?**

This is rarer (~2% of NPIs) — usually happens when CMS reclassifies a specialty mid-year or when a provider legitimately practices under two specialties. Same logic as location: I'm picking the specialty with the most total services billed. Documented this choice rather than silently picking the first one alphabetically.

**Question 3: How do I handle suppressed values in the aggregations?**

CMS suppresses small cells (services with fewer than 11 beneficiaries) by setting `tot_benes` to null while leaving the spending columns populated. The spending is still real money — it just can't be attributed to a beneficiary count. My approach: sum spending columns including suppressed rows (the money was spent), but sum beneficiary counts only across non-suppressed rows. The provider-level beneficiary count will therefore be a slight undercount, which I'll note in the limitations section of the final report.

### What this table doesn't try to do

I'm deliberately not joining to hospital data here. That bridge is its own table (`silver.npi_to_ccn_bridge`) for two reasons:
1. The bridge logic is complex enough to deserve its own dedicated table rather than being baked into the providers join.
2. Many providers (anyone not affiliated with a hospital) won't have a CCN match, and I want unmatched providers to still have a complete row in `silver.providers`.

In [0]:
# Build silver.providers — one row per NPI
# Aggregate MUP-PHY from NPI × HCPCS × place-of-service down to NPI level.

bronze_mup = spark.table(f"{BRONZE_SCHEMA}.medicare_physician_payments")

# Step 1: For each NPI, find the (state, city, zip) combination with the most
# total services. This becomes the provider's "primary location."
location_window = Window.partitionBy("rndrng_npi").orderBy(F.col("loc_total_srvcs").desc())

primary_location = (
    bronze_mup
    .groupBy(
        "rndrng_npi",
        "rndrng_prvdr_state_abrvtn",
        "rndrng_prvdr_city",
        "rndrng_prvdr_zip5",
    )
    .agg(F.sum("tot_srvcs").alias("loc_total_srvcs"))
    .withColumn("rn", F.row_number().over(location_window))
    .filter(F.col("rn") == 1)
    .select(
        "rndrng_npi",
        F.col("rndrng_prvdr_state_abrvtn").alias("primary_state"),
        F.col("rndrng_prvdr_city").alias("primary_city"),
        F.col("rndrng_prvdr_zip5").alias("primary_zip5"),
    )
)

# Step 2: For each NPI, find the specialty with the most total services.
specialty_window = Window.partitionBy("rndrng_npi").orderBy(F.col("spec_total_srvcs").desc())

primary_specialty = (
    bronze_mup
    .groupBy("rndrng_npi", "rndrng_prvdr_type")
    .agg(F.sum("tot_srvcs").alias("spec_total_srvcs"))
    .withColumn("rn", F.row_number().over(specialty_window))
    .filter(F.col("rn") == 1)
    .select(
        "rndrng_npi",
        F.col("rndrng_prvdr_type").alias("primary_specialty"),
    )
)

# Step 3: Identifying info — for each NPI, pick one row's name/credentials
# (these should be identical across rows for the same NPI, but we take the first
# defensively in case of CMS data inconsistencies).
identity = (
    bronze_mup
    .groupBy("rndrng_npi")
    .agg(
        F.first("rndrng_prvdr_last_org_name", ignorenulls=True).alias("last_or_org_name"),
        F.first("rndrng_prvdr_first_name",   ignorenulls=True).alias("first_name"),
        F.first("rndrng_prvdr_mi",           ignorenulls=True).alias("middle_initial"),
        F.first("rndrng_prvdr_crdntls",      ignorenulls=True).alias("credentials"),
        F.first("rndrng_prvdr_ent_cd",       ignorenulls=True).alias("entity_code"),
    )
)

# Step 4: Spending and volume aggregates at the NPI level.
# Spending: summed over all rows (including suppressed-bene rows — the money
# was still spent, even if the bene count is null).
# Beneficiary count: summed only over non-null rows (suppression means the
# count is unknown, not zero).
aggregates = (
    bronze_mup
    .groupBy("rndrng_npi")
    .agg(
        F.sum(F.col("tot_srvcs") * F.col("avg_mdcr_pymt_amt"))
            .alias("total_medicare_payment"),
        F.sum(F.col("tot_srvcs") * F.col("avg_mdcr_alowd_amt"))
            .alias("total_medicare_allowed"),
        F.sum(F.col("tot_srvcs") * F.col("avg_sbmtd_chrg"))
            .alias("total_submitted_charges"),
        F.sum("tot_srvcs").alias("total_services"),
        F.sum(F.when(F.col("tot_benes").isNotNull(), F.col("tot_benes")))
            .alias("total_beneficiaries_unsuppressed"),
        F.countDistinct("hcpcs_cd").alias("unique_hcpcs_codes"),
        F.count("*").alias("source_row_count"),
    )
)

# Step 5: Join everything together
silver_providers = (
    identity
    .join(primary_location,  on="rndrng_npi", how="left")
    .join(primary_specialty, on="rndrng_npi", how="left")
    .join(aggregates,        on="rndrng_npi", how="left")
    .withColumnRenamed("rndrng_npi", "npi")
)

# Lineage column
silver_providers = silver_providers.withColumn("_built_at", F.current_timestamp())

# Write as Delta
(
    silver_providers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("providers")
)

provider_count = spark.table("providers").count()
print(f"Wrote {provider_count:,} rows to {CATALOG}.{SILVER_SCHEMA}.providers")

Wrote 1,175,281 rows to medicare_provider_quality.silver.providers


In [0]:
# Verify silver.providers — sanity-check a known provider and look at distribution

# 1. Pick a known high-volume Virginia provider and see their aggregated view
spark.sql("""
    SELECT
        npi,
        last_or_org_name,
        first_name,
        credentials,
        primary_specialty,
        primary_state,
        primary_city,
        total_services,
        total_medicare_payment,
        unique_hcpcs_codes
    FROM providers
    WHERE primary_state = 'VA'
      AND entity_code = 'I'  -- individuals only, not organizations
    ORDER BY total_medicare_payment DESC NULLS LAST
    LIMIT 5
""").show(truncate=False)

# 2. Sanity check: total payments at the silver level should be in the right
# ballpark vs the underlying bronze
totals = spark.sql(f"""
    SELECT
        SUM(total_medicare_payment) AS silver_total_payment_b,
        SUM(total_services) AS silver_total_services_m
    FROM providers
""").collect()[0]

print(f"\nSilver total Medicare payment: ${totals['silver_total_payment_b']:,.0f}")
print(f"Silver total services (M):     {totals['silver_total_services_m'] / 1_000_000:.1f}M")

+----------+----------------+----------+-----------+------------------+-------------+------------+--------------+----------------------+------------------+
|npi       |last_or_org_name|first_name|credentials|primary_specialty |primary_state|primary_city|total_services|total_medicare_payment|unique_hcpcs_codes|
+----------+----------------+----------+-----------+------------------+-------------+------------+--------------+----------------------+------------------+
|1659355170|Brennan         |Robert    |MD         |Infectious Disease|VA           |Lynchburg   |1081020.0     |1.5926130540039955E7  |40                |
|1295709699|Byrnes          |Gordon    |M.D.       |Ophthalmology     |VA           |Alexandria  |41354.0       |4909767.809992912     |26                |
|1073576377|O'keefe         |John      |M.D.       |Ophthalmology     |VA           |Richmond    |23013.0       |4812250.290002155     |17                |
|1245234715|Byrnes          |Timothy   |MD         |Ophthalmolog

## 2. silver.provider_services — one row per NPI × HCPCS × place-of-service

The line-item table. Same grain as `bronze.medicare_physician_payments`, but cleaned for analytical use.

### Why this table exists if it has the same grain as bronze

Bronze is the durable copy of source data — it should stay close to what CMS shipped. Silver is what analysts actually query. If I push every analytical query against bronze, I either keep repeating the same cleaning logic in every query, or I tempt myself to "fix" bronze and break the principle that bronze is replayable. A separate silver line-item table lets me apply the cleaning once and have downstream gold queries read from a stable, analyst-friendly source.

### What I'm doing in this table

Five things, all small:

1. **Renaming columns** to drop the `rndrng_prvdr_` prefix that CMS uses. `rndrng_prvdr_state_abrvtn` becomes just `provider_state`. The prefix made sense in CMS's context (their dataset has multiple provider entities); in this project there's only one provider per row, so the prefix is noise.

2. **Computing total dollar columns** at the row level. Bronze has `avg_mdcr_pymt_amt` × `tot_srvcs`; I'm pre-computing `total_payment_amt` so every downstream query doesn't have to redo that multiplication.

3. **Dropping columns I won't use** — `rndrng_prvdr_mdcr_prtcptg_ind` (Medicare participation indicator, redundant since everyone in this file participates), credentials, RUCA codes. These can be added back later if needed; cutting them now keeps silver focused.

4. **Adding a `place_of_srvc` flag column** that decodes the cryptic `F`/`O` code into readable `facility`/`non_facility`. This is the kind of denormalization that's the whole point of silver — make the data legible.

5. **Filtering out the country-not-US rows.** A small number of rows in MUP-PHY are for providers in territories or with unknown country codes. For a nationwide analysis these are fine; for any state-level rollups they'd cause issues. Filtering at silver rather than per-query keeps gold logic cleaner.

### What I'm deliberately not doing

- Not deduplicating. Bronze already has correct grain — no dedup needed.
- Not joining to `silver.providers`. That's a denormalization decision I'm deferring to gold. Some analyses want both line-item detail AND provider-level totals; joining them prematurely loses that flexibility.

In [0]:
# Build silver.provider_services — cleaned line-item version of MUP-PHY

source = spark.table(f"{BRONZE_SCHEMA}.medicare_physician_payments")

silver_provider_services = (
    source
    # Filter to US-based providers only
    .filter(F.col("rndrng_prvdr_cntry") == "US")

    # Select and rename columns
    .select(
        F.col("rndrng_npi").alias("npi"),
        F.col("rndrng_prvdr_state_abrvtn").alias("provider_state"),
        F.col("rndrng_prvdr_city").alias("provider_city"),
        F.col("rndrng_prvdr_zip5").alias("provider_zip5"),
        F.col("rndrng_prvdr_type").alias("provider_specialty"),
        F.col("rndrng_prvdr_ent_cd").alias("entity_code"),
        F.col("hcpcs_cd").alias("hcpcs_code"),
        F.col("hcpcs_desc").alias("hcpcs_description"),
        F.col("hcpcs_drug_ind").alias("is_drug_code"),
        F.col("place_of_srvc").alias("place_of_service_code"),
        F.col("tot_benes").alias("beneficiary_count"),
        F.col("tot_srvcs").alias("service_count"),
        F.col("tot_bene_day_srvcs").alias("bene_day_services"),
        F.col("avg_sbmtd_chrg").alias("avg_submitted_charge"),
        F.col("avg_mdcr_alowd_amt").alias("avg_medicare_allowed"),
        F.col("avg_mdcr_pymt_amt").alias("avg_medicare_payment"),
        F.col("avg_mdcr_stdzd_amt").alias("avg_medicare_standardized"),
    )

    # Pre-compute total dollar columns (per the markdown explanation above)
    .withColumn("total_submitted_charge",      F.col("service_count") * F.col("avg_submitted_charge"))
    .withColumn("total_medicare_allowed",      F.col("service_count") * F.col("avg_medicare_allowed"))
    .withColumn("total_medicare_payment",      F.col("service_count") * F.col("avg_medicare_payment"))
    .withColumn("total_medicare_standardized", F.col("service_count") * F.col("avg_medicare_standardized"))

    # Decode place-of-service code into readable label
    .withColumn(
        "place_of_service",
        F.when(F.col("place_of_service_code") == "F", "facility")
         .when(F.col("place_of_service_code") == "O", "non_facility")
         .otherwise("unknown")
    )

    # Lineage
    .withColumn("_built_at", F.current_timestamp())
)

# Write
(
    silver_provider_services.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("provider_services")
)

count = spark.table("provider_services").count()
print(f"Wrote {count:,} rows to {CATALOG}.{SILVER_SCHEMA}.provider_services")

Wrote 9,660,252 rows to medicare_provider_quality.silver.provider_services


In [0]:
# Sanity check silver.provider_services
spark.sql("""
    SELECT 
        npi,
        provider_specialty,
        provider_state,
        hcpcs_code,
        place_of_service,
        service_count,
        total_medicare_payment
    FROM provider_services
    WHERE provider_state = 'VA'
      AND hcpcs_code = '99213'  -- common office visit code
    ORDER BY total_medicare_payment DESC
    LIMIT 5
""").show(truncate=False)

+----------+------------------+--------------+----------+----------------+-------------+----------------------+
|npi       |provider_specialty|provider_state|hcpcs_code|place_of_service|service_count|total_medicare_payment|
+----------+------------------+--------------+----------+----------------+-------------+----------------------+
|1053472159|Internal Medicine |VA            |99213     |non_facility    |5619.0       |436614.160002738      |
|1841290897|Dermatology       |VA            |99213     |non_facility    |5948.0       |364682.690002264      |
|1225149859|Family Practice   |VA            |99213     |non_facility    |2987.0       |238944.260000513      |
|1851399117|Family Practice   |VA            |99213     |non_facility    |3206.0       |216880.730001194      |
|1174704183|Dermatology       |VA            |99213     |non_facility    |3062.0       |190444.280000786      |
+----------+------------------+--------------+----------+----------------+-------------+----------------

## 3. silver.hospitals — one row per CCN (cleaned hospital reference)

The hospital reference table. Same grain as `bronze.hospital_info` (one row per Medicare-certified hospital, keyed on CCN), but cleaned for analytical use and prepared for address-based matching against providers.

### Why this table exists if it has the same grain as bronze

Two reasons:

1. **The bronze schema reflects what CMS publishes — 38 columns including 18 columns of "measure group counts" that are pre-aggregated rollups of the hospital quality measures.** I'm not using those rollups; I'm building my own from the individual measure data in `silver.hospital_quality`. Pruning them at silver keeps the schema focused.

2. **I need normalized address fields for the NPI ↔ CCN bridge.** CMS publishes addresses with inconsistent capitalization ("Dothan" vs "DOTHAN"), trailing whitespace, and some entries that need standardization before fuzzy matching will work. Doing this once here means the bridge query doesn't have to redo it.

### What I'm doing

1. **Keeping only the columns I actually need.** Identity (CCN, name, address, type, ownership), the overall star rating, and the safety/emergency flags. Dropping the 18 measure group count columns — they're rollups of data I have at finer grain in the quality silver table.

2. **Normalizing addresses for matching.** Uppercase everything, strip whitespace, drop common stop-words from street names ("STREET" → "ST", "AVENUE" → "AVE") so the fuzzy match has cleaner input. I'm keeping both the original and normalized address columns — original for display, normalized for matching.

3. **Filtering to Acute Care Hospitals only.** Hospital General Info includes psychiatric hospitals, children's hospitals, and critical access hospitals. For the cost-vs-quality analysis the project is about, only acute care hospitals have the right kind of physician affiliation data and the right kind of quality measure coverage. Filtering at silver keeps gold queries simple and consistent.

### Tradeoff I'm accepting

Filtering to acute care drops about 1,200 hospitals — roughly 22% of the file. The analysis loses coverage of behavioral health and pediatric facilities. I'll flag this in the limitations section of the final report. The alternative (keeping all hospital types and filtering per-query) would scatter the same filter across every gold table and risk inconsistency.

In [0]:
# Build silver.hospitals — cleaned hospital reference

source = spark.table(f"{BRONZE_SCHEMA}.hospital_info")

# Helper UDF-free address normalizer using only built-in functions
def normalize_address_col(col):
    return (
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.upper(F.trim(col)),
                        r"\bSTREET\b", "ST"
                    ),
                    r"\bAVENUE\b", "AVE"
                ),
                r"\bBOULEVARD\b", "BLVD"
            ),
            r"\s+", " "  # collapse multiple spaces
        )
    )

silver_hospitals = (
    source
    # Filter to acute care hospitals only
    .filter(F.col("hospital_type") == "Acute Care Hospitals")
    
    .select(
        F.col("facility_id").alias("ccn"),
        F.col("facility_name").alias("hospital_name"),
        F.col("address").alias("address_original"),
        F.col("city_town").alias("city_original"),
        F.col("state"),
        F.col("zip_code").alias("zip5"),
        F.col("county_parish").alias("county"),
        F.col("telephone_number").alias("phone"),
        F.col("hospital_type"),
        F.col("hospital_ownership"),
        F.col("emergency_services"),
        F.col("hospital_overall_rating"),
    )
    
    # Normalized fields for matching
    .withColumn("address_normalized", normalize_address_col(F.col("address_original")))
    .withColumn("city_normalized",    F.upper(F.trim(F.col("city_original"))))
    
    .withColumn("_built_at", F.current_timestamp())
)

# Write
(
    silver_hospitals.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hospitals")
)

count = spark.table("hospitals").count()
print(f"Wrote {count:,} rows to {CATALOG}.{SILVER_SCHEMA}.hospitals")
print(f"(Bronze had ~5,432 rows; difference is non-acute-care hospitals filtered out)")

Wrote 3,115 rows to medicare_provider_quality.silver.hospitals
(Bronze had ~5,432 rows; difference is non-acute-care hospitals filtered out)


In [0]:
# Sanity check silver.hospitals
spark.sql("""
    SELECT 
        ccn,
        hospital_name,
        state,
        city_original,
        address_original,
        address_normalized,
        hospital_overall_rating,
        hospital_ownership
    FROM hospitals
    WHERE state = 'VA'
      AND hospital_overall_rating = 5
    LIMIT 5
""").show(truncate=False)

# Distribution check
spark.sql("""
    SELECT 
        hospital_overall_rating,
        COUNT(*) AS hospital_count
    FROM hospitals
    GROUP BY hospital_overall_rating
    ORDER BY hospital_overall_rating
""").show()

+------+-----------------------------+-----+-------------+------------------------+------------------------+-----------------------+------------------------------+
|ccn   |hospital_name                |state|city_original|address_original        |address_normalized      |hospital_overall_rating|hospital_ownership            |
+------+-----------------------------+-----+-------------+------------------------+------------------------+-----------------------+------------------------------+
|490004|SENTARA RMH MEDICAL CENTER   |VA   |HARRISONBURG |2010 HEALTH CAMPUS DRIVE|2010 HEALTH CAMPUS DRIVE|5                      |Voluntary non-profit - Private|
|490040|INOVA ALEXANDRIA HOSPITAL    |VA   |ALEXANDRIA   |4320 SEMINARY RD        |4320 SEMINARY RD        |5                      |Voluntary non-profit - Private|
|490043|INOVA LOUDOUN HOSPITAL       |VA   |LEESBURG     |44045 RIVERSIDE PARKWAY |44045 RIVERSIDE PARKWAY |5                      |Voluntary non-profit - Private|
|490059|BON SECO

## 4. silver.hospital_quality — one row per CCN × measure (unified quality measures)

The unified hospital quality table. Combines the three quality bronze tables (`complications_and_deaths`, `unplanned_hospital_visits`, `hcahps`) into one long-format table filtered to the ~10 high-signal measures committed to in Phase 1.

### Why one table instead of three

The three quality bronzes have similar structure (hospital × measure × score) but slightly different columns. `hcahps` has extra fields for star rating and linear mean; the other two have confidence intervals. If I leave them as three separate tables, every gold query has to UNION ALL them and remember which columns exist where. Doing the union once at silver — with a `source_dataset` column so the lineage stays clear — means downstream queries just ask "what's the score for measure X at hospital Y?" against one place.

### The ~10 measures I'm keeping

The Phase 1 plan committed to ~10 high-signal measures. Here's what I'm filtering down to and why:

**Mortality (Complications and Deaths):**
- `MORT_30_AMI` — 30-day mortality after heart attack
- `MORT_30_HF` — 30-day mortality after heart failure
- `MORT_30_PN` — 30-day mortality after pneumonia
- `MORT_30_COPD` — 30-day mortality after COPD
- `MORT_30_STK` — 30-day mortality after stroke
- `MORT_30_CABG` — 30-day mortality after coronary artery bypass surgery

**Readmissions (Unplanned Hospital Visits):**
- `READM_30_AMI` — 30-day readmission after heart attack
- `READM_30_HF` — 30-day readmission after heart failure
- `READM_30_PN` — 30-day readmission after pneumonia
- `Hybrid_HWR` — hospital-wide all-cause 30-day readmission

**Patient Experience (HCAHPS):**
- `H_HSP_RATING_9_10` — patients giving overall hospital rating of 9 or 10
- `H_RECMND_DY` — patients who would definitely recommend the hospital

That's 12 measures, not 10 — the Phase 1 plan said "~10" and I think the readmission and patient experience coverage is worth the extra two. Documented the slight scope expansion explicitly here rather than silently widening the list.

### About `Hybrid_HWR`

The Phase 1 plan listed `READM_30_HOSP_WIDE` as the hospital-wide readmission measure. When I queried bronze in Phase 2, that measure code didn't exist — CMS has replaced it with `Hybrid_HWR` ("Hybrid Hospital-Wide All-Cause Readmission Measure"), which uses claims data combined with EHR data. Same conceptual measure, refreshed methodology. I'm using the current code (`Hybrid_HWR`) since that's what the data actually contains.

### Normalizing the schema across the three sources

The three quality bronzes have different columns. To unify them, I'm selecting a common set:

| Common column | Comes from |
|---|---|
| `ccn` | `facility_id` (all three) |
| `measure_id` | `measure_id` (complications, unplanned) or `hcahps_measure_id` (hcahps) |
| `measure_name` | `measure_name` (complications, unplanned) or `hcahps_question` (hcahps) |
| `score` | `score` (complications, unplanned) or `hcahps_answer_percent` (hcahps, cast to double) |
| `denominator` | `denominator` (complications, unplanned) or `number_of_completed_surveys` (hcahps) |
| `compared_to_national` | `compared_to_national` (complications, unplanned), null for hcahps |
| `start_date` / `end_date` | same name in all three |
| `source_dataset` | a new column I'm adding to make the source traceable |

HCAHPS will be slightly lossy because the source has more nuance (star ratings, linear means, answer breakdowns) than this common schema captures. For the two HCAHPS measures I'm keeping, the `answer_percent` field is the right value to surface — that's what CMS uses for hospital-level comparisons.

In [0]:
# Build silver.hospital_quality — unified quality measures across three bronze sources

# The measure IDs I'm keeping (defined in markdown above)
TARGET_MEASURES_COMPLICATIONS = [
    "MORT_30_AMI", "MORT_30_HF", "MORT_30_PN",
    "MORT_30_COPD", "MORT_30_STK", "MORT_30_CABG",
]
TARGET_MEASURES_UNPLANNED = [
    "READM_30_AMI", "READM_30_HF", "READM_30_PN", "Hybrid_HWR",
]
TARGET_MEASURES_HCAHPS = [
    "H_HSP_RATING_9_10", "H_RECMND_DY",
]

# --- Source 1: Complications and Deaths ---
df_complications = (
    spark.table(f"{BRONZE_SCHEMA}.complications_and_deaths")
    .filter(F.col("measure_id").isin(TARGET_MEASURES_COMPLICATIONS))
    .select(
        F.col("facility_id").alias("ccn"),
        F.col("measure_id"),
        F.col("measure_name"),
        F.col("score"),
        F.col("denominator"),
        F.col("compared_to_national"),
        F.col("start_date"),
        F.col("end_date"),
        F.lit("complications_and_deaths").alias("source_dataset"),
    )
)

# --- Source 2: Unplanned Hospital Visits ---
df_unplanned = (
    spark.table(f"{BRONZE_SCHEMA}.unplanned_hospital_visits")
    .filter(F.col("measure_id").isin(TARGET_MEASURES_UNPLANNED))
    .select(
        F.col("facility_id").alias("ccn"),
        F.col("measure_id"),
        F.col("measure_name"),
        F.col("score"),
        F.col("denominator"),
        F.col("compared_to_national"),
        F.col("start_date"),
        F.col("end_date"),
        F.lit("unplanned_hospital_visits").alias("source_dataset"),
    )
)

# --- Source 3: HCAHPS ---
# HCAHPS has different column names; mapping to the common schema
df_hcahps = (
    spark.table(f"{BRONZE_SCHEMA}.hcahps")
    .filter(F.col("hcahps_measure_id").isin(TARGET_MEASURES_HCAHPS))
    # HCAHPS has multiple answer breakdowns per question; for the rating measures
    # we want, the "headline" row is the one where the answer_percent is populated
    # (not the response rate or other meta-rows). Filter accordingly.
    .filter(F.col("hcahps_answer_percent").isNotNull())
    .select(
        F.col("facility_id").alias("ccn"),
        F.col("hcahps_measure_id").alias("measure_id"),
        F.col("hcahps_question").alias("measure_name"),
        F.col("hcahps_answer_percent").cast(DoubleType()).alias("score"),
        F.col("number_of_completed_surveys").cast(DoubleType()).alias("denominator"),
        F.lit(None).cast(StringType()).alias("compared_to_national"),
        F.col("start_date"),
        F.col("end_date"),
        F.lit("hcahps").alias("source_dataset"),
    )
)

# --- Union the three sources ---
silver_hospital_quality = (
    df_complications
    .unionByName(df_unplanned)
    .unionByName(df_hcahps)
    .withColumn("_built_at", F.current_timestamp())
)

# Write
(
    silver_hospital_quality.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("hospital_quality")
)

count = spark.table("hospital_quality").count()
print(f"Wrote {count:,} rows to {CATALOG}.{SILVER_SCHEMA}.hospital_quality")

# Breakdown by source
spark.sql("""
    SELECT source_dataset, COUNT(*) AS row_count, COUNT(DISTINCT measure_id) AS distinct_measures
    FROM hospital_quality
    GROUP BY source_dataset
    ORDER BY source_dataset
""").show()

Wrote 55,832 rows to medicare_provider_quality.silver.hospital_quality
+--------------------+---------+-----------------+
|      source_dataset|row_count|distinct_measures|
+--------------------+---------+-----------------+
|complications_and...|    28752|                6|
|              hcahps|     7912|                2|
|unplanned_hospita...|    19168|                4|
+--------------------+---------+-----------------+



In [0]:
# Verify the unified table works as a single source
spark.sql("""
    SELECT 
        ccn,
        measure_id,
        score,
        denominator,
        source_dataset
    FROM hospital_quality
    WHERE ccn = '490004'   -- Sentara RMH Medical Center (5-star VA hospital from earlier)
    ORDER BY source_dataset, measure_id
""").show(truncate=False)

+------+-----------------+-----+-----------+-------------------------+
|ccn   |measure_id       |score|denominator|source_dataset           |
+------+-----------------+-----+-----------+-------------------------+
|490004|MORT_30_AMI      |9.6  |210.0      |complications_and_deaths |
|490004|MORT_30_CABG     |2.8  |84.0       |complications_and_deaths |
|490004|MORT_30_COPD     |8.3  |143.0      |complications_and_deaths |
|490004|MORT_30_HF       |10.3 |862.0      |complications_and_deaths |
|490004|MORT_30_PN       |13.2 |636.0      |complications_and_deaths |
|490004|MORT_30_STK      |13.0 |231.0      |complications_and_deaths |
|490004|H_HSP_RATING_9_10|72.0 |603.0      |hcahps                   |
|490004|H_RECMND_DY      |64.0 |603.0      |hcahps                   |
|490004|Hybrid_HWR       |14.9 |2938.0     |unplanned_hospital_visits|
|490004|READM_30_AMI     |12.4 |210.0      |unplanned_hospital_visits|
|490004|READM_30_HF      |20.1 |976.0      |unplanned_hospital_visits|
|49000

## 5. silver.npi_to_ccn_bridge — high-confidence NPI ↔ CCN matches

The most engineering-heavy table in silver. Bridges the two key namespaces: physician data uses NPI, hospital data uses CCN. There's no clean join between them in CMS source data, so this table is derived through address matching.

### The problem

`silver.providers` has 1.17M NPIs with a `primary_state`, `primary_city`, and `primary_zip5`. `silver.hospitals` has 3,115 CCNs with their addresses. To answer questions like "how does this hospital's quality correlate with the spending of physicians who work there?" I need a join key between these two tables — and CMS doesn't publish one.

### The approach I'm taking: Tier 1 (exact match) only

I'm matching providers to hospitals **only where**:
- Provider's `primary_zip5` equals hospital's `zip5`, AND
- Provider's enhanced-normalized address exactly equals hospital's enhanced-normalized address

Anything else is left unmatched. The bridge represents high-confidence affiliations only.

### Alternatives I considered and rejected

**Multi-tier fuzzy matching (Levenshtein-based).** Considered building three tiers: exact match, fuzzy match on same zip, zip-only fallback. Rejected for two reasons:

1. Levenshtein on addresses is a known mediocre matcher. It scores "1234 Main St" vs "1234 Main Street" as a *high* distance (a miss), while scoring "1234 Main St" vs "1235 Main St" as a *low* distance (a false positive). Threshold tuning becomes a guessing game, and the resulting matches look defensible while being systematically wrong in ways that are hard to audit.

2. Doing address fuzzy matching properly requires real address standardization (libpostal, USPS validation, or commercial geocoding APIs). Halfway with Levenshtein is worse than not doing it at all — it adds confidence I haven't earned.

**Using the NPPES NPI registry as a third data source.** NPPES doesn't actually publish NPI-to-CCN relationships — the public registry only has addresses, same as I already have. The CCN-to-NPI linkage exists in CMS PECOS, which has restricted access. NPPES would add a 10 GB download for marginal improvement in address quality.

**No bridge table at all.** Would mean dropping the cost-vs-quality-at-hospital-level analysis entirely. That's a real loss for the portfolio narrative, and Tier 1 exact match is a defensible engineering compromise.

### The enhanced address normalization

Standard normalization (uppercase, trim, expand STREET→ST) handles the easy cases. For the bridge I'm adding a step: strip out suite/floor/room/building indicators from provider addresses before comparing. Hospital addresses rarely have these suffixes; provider addresses commonly do. Stripping recovers matches that would otherwise fail.

Examples:
- `"100 MEDICAL CENTER DR STE 200"` → `"100 MEDICAL CENTER DR"`
- `"500 HOSPITAL BLVD FLOOR 3"` → `"500 HOSPITAL BLVD"`
- `"123 MAIN ST BLDG A"` → `"123 MAIN ST"`

### Expected match rate

Realistically, 5–15% of the 1.17M NPIs will match. That's not a bug — most physicians (primary care, outpatient specialists, retail clinics) don't work at hospitals at all and *shouldn't* match. The match rate is mostly a function of how many hospital-affiliated physicians are in the data, not how good my matching is.

### What downstream tables can and can't do with this bridge

**Can do:** join physician spending data to hospital quality data for the subset of providers who matched. Gold tables like `hospital_value_scorecard` will be built on this subset.

**Can't do:** make claims about "all hospital-affiliated physicians." The bridge is conservative; it under-represents real affiliations. Any hospital-level analysis built on it will be biased toward providers with hospital street addresses in MUP-PHY.

I'll surface this limitation explicitly in the final report.

In [0]:
# Build silver.npi_to_ccn_bridge — Tier 1 exact match with enhanced normalization

# Enhanced normalizer that also strips suite/floor/room/building indicators
def enhanced_normalize(col):
    return (
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.regexp_replace(
                            F.regexp_replace(
                                F.regexp_replace(
                                    F.upper(F.trim(col)),
                                    # Expand common street suffixes
                                    r"\bSTREET\b", "ST"
                                ),
                                r"\bAVENUE\b", "AVE"
                            ),
                            r"\bBOULEVARD\b", "BLVD"
                        ),
                        # Strip suite/floor/room/building indicators and what follows
                        r"\s+(STE|SUITE|FL|FLR|FLOOR|RM|ROOM|BLDG|BUILDING|UNIT|APT|#)\s*[A-Z0-9\-]+\s*$",
                        ""
                    ),
                    # Collapse multiple spaces
                    r"\s+", " "
                ),
                # Trim trailing punctuation/spaces
                r"\s*\.?\s*$", ""
            ),
            # Final whitespace trim
            r"^\s+|\s+$", ""
        )
    )

# Get providers with their first practice location address from bronze
# (silver.providers only has primary city/state/zip — we need street too)
# So we need to re-derive the most-used street address per NPI
location_window = Window.partitionBy("rndrng_npi").orderBy(F.col("loc_total_srvcs").desc())

provider_addresses = (
    spark.table(f"{BRONZE_SCHEMA}.medicare_physician_payments")
    .filter(F.col("rndrng_prvdr_cntry") == "US")
    .groupBy(
        "rndrng_npi",
        "rndrng_prvdr_st1",
        "rndrng_prvdr_zip5",
    )
    .agg(F.sum("tot_srvcs").alias("loc_total_srvcs"))
    .withColumn("rn", F.row_number().over(location_window))
    .filter(F.col("rn") == 1)
    .select(
        F.col("rndrng_npi").alias("npi"),
        F.col("rndrng_prvdr_st1").alias("provider_street"),
        F.col("rndrng_prvdr_zip5").alias("provider_zip5"),
    )
    # Take only the first 5 digits in case CMS stored ZIP+4
    .withColumn("provider_zip5", F.substring(F.col("provider_zip5"), 1, 5))
    .withColumn(
        "provider_street_normalized",
        enhanced_normalize(F.col("provider_street"))
    )
    .filter(F.col("provider_street_normalized").isNotNull())
    .filter(F.col("provider_street_normalized") != "")
)

# Get hospital addresses with the same enhanced normalization
hospital_addresses = (
    spark.table("hospitals")
    .select(
        F.col("ccn"),
        F.col("hospital_name"),
        F.col("address_original").alias("hospital_street"),
        F.col("zip5").alias("hospital_zip5"),
    )
    .withColumn("hospital_zip5", F.substring(F.col("hospital_zip5"), 1, 5))
    .withColumn(
        "hospital_street_normalized",
        enhanced_normalize(F.col("hospital_street"))
    )
)

# Tier 1 exact match: same zip5 AND same normalized street
bridge = (
    provider_addresses.alias("p")
    .join(
        hospital_addresses.alias("h"),
        on=[
            F.col("p.provider_zip5") == F.col("h.hospital_zip5"),
            F.col("p.provider_street_normalized") == F.col("h.hospital_street_normalized"),
        ],
        how="inner",
    )
    .select(
        F.col("p.npi"),
        F.col("h.ccn"),
        F.col("h.hospital_name"),
        F.col("p.provider_street").alias("matched_provider_address"),
        F.col("h.hospital_street").alias("matched_hospital_address"),
        F.col("p.provider_zip5").alias("matched_zip5"),
        F.lit("tier_1_exact").alias("match_confidence"),
    )
    .withColumn("_built_at", F.current_timestamp())
)

# Handle the multi-match case: same address can house multiple billing entities.
# A single NPI matching multiple CCNs (multi-campus hospitals) is rare but legitimate.
# A single CCN matching multiple NPIs at the same address is normal (it's a hospital).
# I'm keeping all matches as-is — analysts can deduplicate per use case.

# Write
(
    bridge.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("npi_to_ccn_bridge")
)

count = spark.table("npi_to_ccn_bridge").count()
distinct_npis = spark.sql("SELECT COUNT(DISTINCT npi) AS n FROM npi_to_ccn_bridge").collect()[0]["n"]
distinct_ccns = spark.sql("SELECT COUNT(DISTINCT ccn) AS n FROM npi_to_ccn_bridge").collect()[0]["n"]

print(f"Wrote {count:,} matches to {CATALOG}.{SILVER_SCHEMA}.npi_to_ccn_bridge")
print(f"  Distinct NPIs matched:  {distinct_npis:,}  ({distinct_npis / 1_175_281 * 100:.1f}% of all providers)")
print(f"  Distinct CCNs matched:  {distinct_ccns:,}  ({distinct_ccns / 3_115 * 100:.1f}% of all hospitals)")

Wrote 109,780 matches to medicare_provider_quality.silver.npi_to_ccn_bridge
  Distinct NPIs matched:  109,309  (9.3% of all providers)
  Distinct CCNs matched:  1,951  (62.6% of all hospitals)


In [0]:
# Verify: who matched to Sentara RMH Medical Center (CCN 490004)?
spark.sql("""
    SELECT 
        b.ccn,
        b.hospital_name,
        b.npi,
        p.last_or_org_name,
        p.first_name,
        p.primary_specialty,
        p.total_medicare_payment
    FROM npi_to_ccn_bridge b
    JOIN providers p ON b.npi = p.npi
    WHERE b.ccn = '490004'
    ORDER BY p.total_medicare_payment DESC NULLS LAST
    LIMIT 10
""").show(truncate=False)

+------+--------------------------+----------+----------------+----------+--------------------+----------------------+
|ccn   |hospital_name             |npi       |last_or_org_name|first_name|primary_specialty   |total_medicare_payment|
+------+--------------------------+----------+----------------+----------+--------------------+----------------------+
|490004|SENTARA RMH MEDICAL CENTER|1992744270|Wagner          |Andrew    |Diagnostic Radiology|170814.1900000569     |
|490004|SENTARA RMH MEDICAL CENTER|1538110135|Buckman         |Peter     |Thoracic Surgery    |64619.040000092       |
|490004|SENTARA RMH MEDICAL CENTER|1811903834|Caldwell        |Roumiana  |Internal Medicine   |45063.789999609995    |
|490004|SENTARA RMH MEDICAL CENTER|1356357305|Rizvi           |Reena     |Family Practice     |44460.489999999       |
|490004|SENTARA RMH MEDICAL CENTER|1780982769|Landacre        |Dana      |Physician Assistant |22341.080000028       |
|490004|SENTARA RMH MEDICAL CENTER|1871532135|Be

In [0]:
# Check specialty value quality — are there inconsistencies worth fixing?

# 1. How many distinct specialty values are there?
spark.sql("""
    SELECT COUNT(DISTINCT primary_specialty) AS distinct_specialties
    FROM providers
""").show()

# 2. Look for whitespace/casing inconsistencies (specialties that look like duplicates)
spark.sql("""
    SELECT primary_specialty, COUNT(*) AS provider_count
    FROM providers
    WHERE primary_specialty IS NOT NULL
    GROUP BY primary_specialty
    ORDER BY primary_specialty
""").show(150, truncate=False)

+--------------------+
|distinct_specialties|
+--------------------+
|                 104|
+--------------------+

+-------------------------------------------------------+--------------+
|primary_specialty                                      |provider_count|
+-------------------------------------------------------+--------------+
|Addiction Medicine                                     |218           |
|Adult Congenital Heart Disease                         |46            |
|Advanced Heart Failure and Transplant Cardiology       |783           |
|All Other Suppliers                                    |52            |
|Allergy/ Immunology                                    |3118          |
|Ambulance Service Provider                             |9332          |
|Ambulatory Surgical Center                             |5353          |
|Anesthesiology                                         |34189         |
|Anesthesiology Assistant                               |2157          |
|Audiolo